# OSRT Ostinato — v7 pretrain on Colab

Target runtime: **RTX PRO 6000 Blackwell, 96GB**.

Run the cells in order. Cell 1 is a gate — if it does not report `sm_120`,
stop and read its verdict before spending anything.

**Sessions are capped and the VM's disk is wiped on release.** Cell 5 sets
`--hf-repo`, which pulls the newest checkpoint before training and pushes
each new one as it is written. Without it a disconnect loses the run.

Do **not** background the training cell (`nohup`, `&`). Colab tears down the
runtime when the foreground cell returns, so a backgrounded trainer dies
with it — run it in the foreground and leave the tab open.


## 1 · Gate: what card is this, and can the expert path go low-precision?

This is also the first half of roadmap gate **G7**. Routed experts are ~84%
of v7's parameters and run through the private `torch._grouped_mm`; if that
refuses FP8, the NVFP4 case in §13.3 covers under a third of the model.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


## 2 · Install


In [ ]:
!git clone -q https://github.com/CodeHalwell/OSRT-Ostinato.git
%cd OSRT-Ostinato
!pip install -q -e . 2>&1 | tail -2
!PYTHONPATH=src python scripts/probe_gpu.py


## 3 · Secrets

Add `HF_TOKEN` (write access to your checkpoint repo) and `WANDB_API_KEY`
in the Colab **Secrets** panel (🔑, left sidebar). They never touch this
notebook or git.


In [ ]:
import os

from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert os.environ['HF_TOKEN'] and os.environ['WANDB_API_KEY'], 'set both in Secrets'
print('secrets set')


## 4 · Build the tokenizer and confirm the shape

`compute_budget.py` is the only trusted source for parameter counts — the
repo deliberately states none in any name. Expect **968,468,355 physical /
263,035,779 active**; anything else means the config drifted.


In [ ]:
!python scripts/build_tokenizer_v7.py --out tokenizer
!PYTHONPATH=src python scripts/compute_budget.py


## 5 · Launch

Set `HF_CKPT_REPO` to a **private** repo you own. Re-running this cell in a
later session resumes from the newest checkpoint it finds there — that is
the whole cross-session mechanism.

`--total-steps` is deliberately small here. Do a short run first and read
the loss and the MoE telemetry before committing real budget.


In [ ]:
HF_CKPT_REPO = 'HallD/osrt-v7-ckpt'   # <-- your private repo

!PYTHONPATH=src python -m osrt.train_main \
    --tokenizer-path ./tokenizer \
    --ckpt-dir ./checkpoints/v7 \
    --hf-repo {HF_CKPT_REPO} \
    --total-steps 200 \
    --wandb-run-name osrt-v7-pretrain-s1


## 6 · What to watch

| signal | healthy | worry |
|---|---|---|
| `loss` | falling, no spikes | flat, or spiking and not recovering |
| `dead_experts` | 0 | anything > 0 — at E=28 each is 3.6% of block capacity |
| `load_entropy` | near ln(28) ≈ 3.33 | falling — router collapse |
| `loop_upd` norms | non-trivial at every loop | last loops → 0 = loop collapse |

Quantile Balancing should hold `dead_experts` at 0 without tuning; it is a
one-shot solve, not an integrator, so it cannot slowly drift.

### Still open before a real trunk run

- **G3a** — does the token requirement track active or total params? Three
  ladder runs at ~150M. It can re-price the 968M shape downward, so a full
  trunk before it reports is a bet.
- **G7 second half** — NVFP4 throughput, once the probe confirms sm_120.
- Peak memory at batch 6 / seq 4096 has never been measured on real hardware.
